# Baseline modeling

**GOAL:**

- Train multiple models
- compare their performance and metrics
- select the promising model

In [1]:
# Imports

import pandas as pd
import numpy as np

In [2]:
# load X_train, X_val, X_test, y_train, y_val, y_test

X_train = pd.read_csv(r'..\Data\processed\modeling\X_train.csv')
X_val = pd.read_csv(r'..\Data\processed\modeling\X_val.csv')
X_test = pd.read_csv(r'..\Data\processed\modeling\X_test.csv')

y_train = pd.read_csv(r'..\Data\processed\modeling\y_train.csv')
y_val = pd.read_csv(r'..\Data\processed\modeling\y_val.csv')
y_test = pd.read_csv(r'..\Data\processed\modeling\y_test.csv')

## 1: Baseline model

- linear regression

**comparison models**

- ridge regression
- random forest
- XG boost

**Comparison**

- compare by RMSE Rsqt

In [44]:
# helper functions
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, root_mean_squared_error

all_results = {}
# all_results = pd.DataFrame(results).T
def log_metrics(y_val, y_pred):

    MAE = mean_absolute_error(y_val, y_pred)
    RMSE = root_mean_squared_error(y_val, y_pred)
    R2 = r2_score(y_val, y_pred)

    return MAE, RMSE, R2

def actual_metrics(y_val, y_pred):

    ACTUAL_Y_VALIDATION = np.expm1(y_val)
    ACTUAL_Y_PRED = np.expm1(y_pred)

    ACUTAL_MAE = mean_absolute_error(ACTUAL_Y_VALIDATION, ACTUAL_Y_PRED)
    ACTUAL_RMSE = root_mean_squared_error(ACTUAL_Y_VALIDATION, ACTUAL_Y_PRED)
    ACTUAL_R2 = r2_score(ACTUAL_Y_VALIDATION, ACTUAL_Y_PRED)

    return ACUTAL_MAE, ACTUAL_RMSE, ACTUAL_R2

def result(model_name, model, X_val, y_val):

    y_pred = model.predict(X_val)
    MAE, RMSE, R2 = log_metrics(y_val, y_pred)
    A_MAE, A_RMSE, A_R2 = actual_metrics(y_val, y_pred)

    all_results[model_name] = {
        'Log MAE': round(MAE, 4),
        'Log RMSE': round(RMSE, 4),
        'Log R2': round(R2, 4),
        'Actual MAE': round(A_MAE, 4),
        'Actual RMSE': round(A_RMSE, 4),
        'Actual R2': round(A_R2, 4)
    }

In [45]:
# 1.1 Baseline model selcetion (Linear regression)
from sklearn.linear_model import LinearRegression

lr = LinearRegression()
lr.fit(X_train, y_train)

result('Linear Regression', lr, X_val, y_val)
pd.DataFrame(all_results).T

,Log MAE,Log RMSE,Log R2,Actual MAE,Actual RMSE,Actual R2
Linear Regression,0.1994,0.303,0.3345,1754.5806,4462.086,0.1646


# Baseline Model Report

## MAE - Error Metric
- Mean Absolute Error in actual price,
    - tells me that my regression model's predictions are **on average, ~1754 rs** away from the actual rent

## RMSE - Error Metric
- RMSE tells a gap,
    ```md
    RMSE = 4462
    MAE  = 1754
    Gap  = 2708  ← this is large
    ```
    - This gap means my model has some bad predictions (outliers), that pulls up RMSE score
    - Most predictions are around ₹1,754 off (MAE)
    - But a few are severely bad pulling RMSE up to ₹4,462
    - The gap (₹2,708) just tells us how much those outliers are skewing the score

## R2 Score - Eval metric

- R2 Score tells me that my baseline model (Linear Regression), can explains about ~33% of variation in the rent
    - so Since R2 score was low, 0.33 is weak for this dataset
- ** why R2 score is low?**
    - Linear regression can not predict non linear predictions

# Final Observation: Linear Regression Baseline

---

| Metric | Score | Indication | Better | Decision |
|--------|-------|------------|--------|----------|
| **MAE** | ₹1,754 | On average, every prediction is ₹1,754 away from the actual rent | ⬇️ Lower = Better | Train another model and compare |
| **RMSE > MAE** | Gap = ₹2,708 | Most predictions are ~₹1,754 off, but a few are severely wrong pulling RMSE up to ₹4,462 | ⬇️ Lower = Better | Train another model and compare |
| **R² Score** | 0.33 | Linear Regression model's prediction is weak ~67% of rent variation unexplained | ⬆️ Higher = Better | Train another model (tree-based) to improve |

---

> **Verdict:** Linear Regression is not good enough. Move to tree-based models (Random Forest / XGBoost) to capture non-linear rent patterns.

In [16]:
lr.coef_

array([[-0.07099695,  0.40381681,  0.44220164,  0.00606746, -0.00652418,
        -0.11000773,  0.13723976, -0.00303758,  0.25176793,  0.02357802,
        -0.09637525,  0.00417878,  0.15154267,  0.02307527, -0.04958345,
         0.03007933,  0.1287062 , -0.26377551,  0.01675561,  0.003946  ,
         0.10853681,  0.07348781, -0.11116017,  0.06455939,  0.06289336,
        -0.0180705 , -0.04482286,  0.04378457, -0.05731983,  0.06514942,
        -0.05161416, -0.03545369,  0.01008463,  0.02536906]])

---
# Train More Models to comapre against baseline
---

## **Ok now before trying tree based models, I'm going to try regularization with the linear model**

### L2 Regularization (Ridge Regression)

- Since i have many featues and corelating with each (some), linear regression may gave too much importance to some features
- The goal is to penalize those coefficients and potentially reduce the variance or overfitting 


In [17]:
X_train.head()

,latitude,longitude,locality,transit_score,lifestyle_score,occupancy,deposit,attached_bathroom,food_included,mess,...,gender_BOTH,gender_FEMALE,gender_MALE,parking_Bike,parking_Bike and Car,parking_Car,parking_No Parking,available_for_Anyone,available_for_Student,available_for_Working Professional
0,12.913368,80.228812,8.935934,6.3,6.3,2.0,8.294300,1,1,1,...,0.0,1.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
1,12.979819,80.242495,8.901274,8.0,7.9,2.0,8.006701,0,1,0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
2,12.989589,80.248599,8.886307,7.9,8.3,0.0,10.283669,0,0,1,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
3,12.926202,80.111700,8.876267,8.2,9.6,2.0,7.601402,0,1,0,...,0.0,0.0,1.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
4,13.053022,80.213763,8.693086,6.7,6.3,2.0,10.308986,0,0,0,...,1.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


In [46]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

# before train a ridge model, i need to scale the features, cuz ridge is sensitive to features scale
# so basically ridge is used with standard scalar

scaler = StandardScaler()

# fit on train only ; fit means learn the rules on the data
scaler.fit(X_train)

# transform on train/test/val ; transform means, apply the learned rules on the data
X_train_scaled = scaler.transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# to find the best alpha value
for alpha in [1, 5, 10, 40, 42, 43,44, 60]:
    ridge = Ridge(alpha=alpha)
    ridge.fit(X_train_scaled, y_train)
    pred = ridge.predict(X_val_scaled)

    print(
        round(mean_absolute_error(y_val, (pred)), 4),
        round(root_mean_squared_error(y_val, pred), 4),
        round(r2_score(y_val, pred), 4)
    )

0.1993 0.303 0.3346
0.1991 0.3029 0.3352
0.1989 0.3027 0.3357
0.1988 0.3025 0.3367
0.1989 0.3025 0.3367
0.1989 0.3025 0.3367
0.1989 0.3025 0.3366
0.199 0.3026 0.3363


In [47]:
# model fit

alpha = 43 # found that the alpha = 43 is penalize value from the above iterations
ridge = Ridge(alpha=alpha)
ridge.fit(X_train_scaled, y_train)
result('Ridge rgression', ridge, X_val_scaled, y_val)
pd.DataFrame(all_results).T

,Log MAE,Log RMSE,Log R2,Actual MAE,Actual RMSE,Actual R2
Linear Regression,0.1994,0.3030,0.3345,1754.5806,4462.0860,0.1646
Ridge rgression,0.1989,0.3025,0.3367,1754.2632,4478.4207,0.1585


## Ridge Regression result:

- alpha = ~43.0 gives the best result = 0.3367
- still similar to the linear regression, no drastical improvements


**Takeaway**: Ridge isn't dramatically better than Linear Regression for this dataset. That's actually a useful modeling finding.

---
# Lasso Regression

- since ridge isnt make any improvement
- here i see lasso, hope it could lead to anything

In [48]:
from sklearn.linear_model import Lasso

for alpha in [0.0001, 0.001, 0.005, 0.006, 0.1, 0.5, 1]:
    lasso = Lasso(alpha=alpha)
    lasso.fit(X_train_scaled, y_train)
    pred = lasso.predict(X_val_scaled)

    print(
        round(mean_absolute_error(y_val, pred), 4),
        round(root_mean_squared_error(y_val, pred), 4),
        round(r2_score(y_val, pred), 4)
    )

0.1992 0.3029 0.3349
0.1986 0.3024 0.3374
0.1986 0.3022 0.3382
0.1986 0.3024 0.3371
0.2473 0.3603 0.0593
0.2581 0.3717 -0.0015
0.2581 0.3717 -0.0015


In [49]:
# here alpha = 0.005 is the best hyperparametr, observed from the above iteration
lasso = Lasso(alpha=0.005)
lasso.fit(X_train_scaled, y_train)
result('lasso', lasso, X_val_scaled, y_val)
# lasso.fit(X_train_scaled, y_train)
# pred = lasso.predict(X_val_scaled)
pd.DataFrame(all_results).T

,Log MAE,Log RMSE,Log R2,Actual MAE,Actual RMSE,Actual R2
Linear Regression,0.1994,0.3030,0.3345,1754.5806,4462.0860,0.1646
Ridge rgression,0.1989,0.3025,0.3367,1754.2632,4478.4207,0.1585
lasso,0.1986,0.3022,0.3382,1754.8490,4482.2114,0.1570


## lasso result:

- Lasso gives me almost the same performance while removing some less useful features.

    ```md 
    Linear Regression → R² ≈ 0.3345
    Ridge             → R² ≈ 0.3367
    Lasso (α=0.005)   → R² ≈ 0.3382
    ```

- so far, **Conclusion**: Lasso is currently your best of the three, but the improvement is small. Now you've learned why Lasso is useful rather than just chasing a higher score.